# LoRA: технический bootstrap, шаги 1–2
Это **не обучение на Food-101 и не LR search**. Проверяем окружение, запись на Drive и PEFT LP/A/M/B на искусственных входах.

1. Выберите Runtime → Change runtime type → GPU.
2. Выполните установку, затем Runtime → Restart session, **после перезапуска продолжайте с загрузки архива**.
3. Загрузите `colab-bootstrap.zip`, полученный вместе с этим notebook.
Версии — bootstrap-кандидат; успешная установка ещё не означает пройденную приёмку.


In [ ]:
%pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu126
%pip install transformers==4.57.1 peft==0.17.1 accelerate==1.10.1 huggingface-hub==0.35.3 safetensors==0.6.2 numpy==2.2.6 Pillow==11.3.0


## После перезапуска: загрузка единой версии кода

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, io, json, zipfile, os
uploaded = files.upload()  # Выбрать только colab-bootstrap.zip.
if len(uploaded) != 1:
    raise RuntimeError("Нужен один архив colab-bootstrap.zip")
root = Path('/content/lora_project')
root.mkdir(exist_ok=False)  # Новая папка: не смешиваем версии.
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in archive.namelist():
        if not (root / name).resolve().is_relative_to(root.resolve()):
            raise RuntimeError("Небезопасный путь в архиве")
    archive.extractall(root)
manifest = json.loads((root / 'SOURCE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    actual = hashlib.sha256((root / name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"Hash mismatch: {name}")
os.chdir(root)
os.environ['HF_HOME'] = '/content/hf-cache'
print('Source verified:', manifest['release_hash'])


## Проверка CUDA и постоянной папки результатов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Для общего хранилища укажите реальный путь выданной вам папки/ярлыка.
PERSISTENT_ROOT = Path('/content/drive/MyDrive/LoRA_experiment')
EXECUTOR_ID = 'participant_1'  # У каждого участника своё значение.
from datetime import datetime, timezone
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out = PERSISTENT_ROOT / 'executors' / EXECUTOR_ID / 'technical' / SESSION_ID
out.mkdir(parents=True, exist_ok=False)
import subprocess, sys
subprocess.run([sys.executable, 'bootstrap.py', '--require-cuda',
                '--storage', str(out), '--output', str(out / 'environment.json')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Реальная структура ViT и PEFT, без скачивания весов
Случайные веса, четыре метода, искусственные изображения 224×224. Проверяются модули, параметры, gradients и frozen weights. Это ещё **не замер вместимости microbatch16**.

In [ ]:
subprocess.run([sys.executable, 'test_params.py', '--device', 'cuda', '--image-size', '224',
                '--output', str(out / 'synthetic_smoke.json')], check=True)


## Замер начального training microbatch для Both
Три обновления по 32 порции из 16 изображений. При OOM сначала завершаем процесс, затем повторяем с 8/64, далее 4/128. Другие ошибки не маскируются под OOM. Validation, Trainer/resume и реальные данные проверяются на следующем этапе; этот замер не является полной приёмкой.

In [ ]:
BATCH, ACCUMULATION = 16, 32
subprocess.run([sys.executable, 'test_params.py', '--config', 'B-iso', '--device', 'cuda',
                '--image-size', '224', '--batch-size', str(BATCH),
                '--accumulation', str(ACCUMULATION),
                '--output', str(out / f'profile_b{BATCH}_a{ACCUMULATION}.json')], check=True)


## Необязательно: загрузка исходных pretrained weights
Разрешён только полный commit SHA. Его один раз разрешаем и сохраняем, затем все участники используют тот же SHA. Этот блок также не обучает Food-101.

In [ ]:
# Выполнить один раз координатору; передать model_revision.json участникам.
from huggingface_hub import HfApi
from configs import MODEL_ID
revision_file = PERSISTENT_ROOT / 'protocol' / 'model_revision.json'
if revision_file.exists():
    model_revision = json.loads(revision_file.read_text())['revision']
else:
    model_revision = HfApi().model_info(MODEL_ID).sha
    revision_file.parent.mkdir(parents=True, exist_ok=True)
    revision_file.write_text(json.dumps({'model': MODEL_ID, 'revision': model_revision}, indent=2))
subprocess.run([sys.executable, 'test_params.py', '--config', 'B-iso', '--device', 'cuda',
                '--image-size', '224', '--pretrained', '--revision', model_revision,
                '--output', str(out / 'pretrained_smoke.json')], check=True)


## Сохранить свидетельство окружения
Список ниже — фактически установленные версии, ещё не окончательный общий lock. Перед завершением сессии убедитесь, что отчёты появились на Drive. Для source/модели и полной серии сохраняются отдельные hashes/revisions.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(out / 'pip-freeze.txt').write_text(result.stdout)
(out / 'SOURCE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
print('Технические отчёты:', out)
